# NeuralEdge: Sleep-EDF sleep staging on Colab

Trains the 1-D CNN on **real** Sleep-EDF Expanded data (sleep-cassette, EEG Fpz-Cz, 30-s epochs, 5 AASM classes) with subject-wise cross-validation, then:

* macro-F1 (mean ± std over folds), per-class F1, Cohen's κ, confusion matrix
* ONNX export + ONNX Runtime **CPU** latency (Colab's cloud CPU, *not* an edge device)
* writes `results/sleepedf_results.json` and a paste-ready `results/sleepedf_results.md`

**Before you start:** Runtime → Change runtime type → **T4 GPU** (CPU works but training is much slower).
The download is about 40 PSG recordings from physionet.org (roughly 2 GB) and goes to `/content/mne_data`.

Reference for comparison: DeepSleepNet (Supratak et al., IEEE TNSRE 2017, [arXiv:1703.04046](https://arxiv.org/abs/1703.04046)) reports **macro-F1 76.9 / accuracy 82.0** on Sleep-EDF Fpz-Cz with 20-fold CV. It is a CNN + BiLSTM sequence model on the 2013 release, so it is a reference point, not a like-for-like baseline.

## 1. Settings

In [ ]:
# Where to get the code: push the repo to GitHub first (or point REPO_URL at a fork),
# or set REPO_URL = None and upload NeuralEdge.zip (git archive) in the next cell.
REPO_URL = "https://github.com/LasyaRamachandruni/NeuralEdge.git"
BRANCH = "main"

SUBJECTS = "0-19"        # SC subjects 0-19 = the "Sleep-EDF-20" subset used by DeepSleepNet
MODEL = "resnet1d_tiny"  # or "mobilenet1d"
FOLDS = 20               # 20 = leave-one-subject-out (same fold count as DeepSleepNet); 5 is ~4x faster
EPOCHS = 30
PATIENCE = 5
RUN_COMPRESSION = False  # QAT + pruning on the deploy fold (CPU, adds time)

## 2. Get the code and install dependencies

In [ ]:
import os, subprocess
os.chdir("/content")
if REPO_URL:
    if not os.path.exists("NeuralEdge"):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "NeuralEdge"], check=True)
else:
    from google.colab import files
    up = files.upload()  # choose NeuralEdge.zip
    zip_name = next(iter(up))
    subprocess.run(["unzip", "-q", "-o", zip_name, "-d", "NeuralEdge"], check=True)
os.chdir("/content/NeuralEdge")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout or "(no git metadata)")

In [ ]:
!pip install -q "mne>=1.6" "onnx>=1.15" "onnxruntime>=1.17" edfio pytest
!nvidia-smi -L || echo "No GPU - training will run on CPU"
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Quick self-check of the pipeline on synthetic data + a tiny fake EDF (about a minute)
!python -m pytest -q

## 3. Download Sleep-EDF and build 30-s epochs
Fpz-Cz at 100 Hz, R&K → AASM (S3+S4 → N3), movement/unscored dropped, wake trimmed to 30 min around sleep, z-scored per recording. Both nights of a subject are stored together so no person is ever in both train and test.

Fast download: PhysioNet also publishes Sleep-EDF on its AWS open-data bucket (`s3://physionet-open`), which is usually far faster than physionet.org. This copies the sleep-cassette files for the chosen subjects into the folder MNE reads from, so the next cell finds them and skips the slow download.

In [ ]:
!pip install -q awscli
lo, hi = (int(x) for x in SUBJECTS.split('-'))
includes = ' '.join(f'--include "SC4{s:02d}*"' for s in range(lo, hi + 1))
!mkdir -p /content/mne_data/physionet-sleep-data
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/sleep-edfx/1.0.0/sleep-cassette/ /content/mne_data/physionet-sleep-data/ --exclude "*" {includes}
!ls /content/mne_data/physionet-sleep-data | wc -l && du -sh /content/mne_data/physionet-sleep-data

In [ ]:
!python data/prepare_sleepedf.py --download --subjects {SUBJECTS} --mne_path /content/mne_data \
    --out data/processed/sleepedf --calib_out data/calib/sleepedf

In [ ]:
import pandas as pd
pd.read_csv("data/processed/sleepedf/meta.csv")

## 4. Train with subject-wise cross-validation
Inside each fold, a few training subjects are held out for early stopping; the test subjects are only scored once at the end. Every fold's checkpoint is saved in `artifacts/`.

In [ ]:
!python train.py --dataset sleepedf --data_root data/processed --model {MODEL} \
    --folds {FOLDS} --epochs {EPOCHS} --early_stop_patience {PATIENCE} --batch_size 128 --lr 1e-3 \
    --num_workers 2 --log_dir results

## 5. Export to ONNX and measure CPU latency
This is Colab's server CPU with ONNX Runtime, batch 1, 1 thread. Report it as such; it is not a Jetson or Raspberry Pi number.

In [ ]:
!python export_onnx.py --model {MODEL} --checkpoint artifacts/sleepedf_{MODEL}_fp32.pt \
    --data_dir data/processed/sleepedf --onnx_path artifacts/sleepedf_{MODEL}.onnx
!python bench.py --mode onnx-cpu --onnx artifacts/sleepedf_{MODEL}.onnx --threads 1 --iters 1000 \
    --label "Colab CPU, trained sleepedf deploy checkpoint" --json_out results/latency_cpu_colab.json

## 6. Optional: QAT (INT8) and pruning on the deploy fold

In [ ]:
if RUN_COMPRESSION:
    !python qat_train.py --dataset sleepedf --data_root data/processed --model {MODEL} --epochs 3 --log_dir results
    !python prune.py --dataset sleepedf --data_root data/processed --model {MODEL} --prune_ratio 0.3 --finetune_epochs 5 --log_dir results
else:
    print("skipped (set RUN_COMPRESSION = True in the settings cell)")

## 7. Evaluation summary: macro-F1, per-class F1, confusion matrix

In [ ]:
!python scripts/summarize_sleepedf.py --model {MODEL} --latency_json results/latency_cpu_colab.json

In [ ]:
import json
from IPython.display import Image, Markdown, display
r = json.load(open("results/sleepedf_results.json"))
print(f"macro-F1 {r['macro_f1_mean']:.3f} ± {r['macro_f1_std']:.3f} over {r['n_folds_run']} folds ({r['cv_scheme']})")
print("per-fold:", [round(f, 3) for f in r["per_fold_macro_f1"]])
print("per-class F1:", {k: round(v, 3) for k, v in r["per_class_f1"].items()})
labels = r["confusion_matrix"]["labels"]
print(pd.DataFrame(r["confusion_matrix"]["rows_true_cols_pred"], index=[f"true {l}" for l in labels], columns=[f"pred {l}" for l in labels]))
display(Image("results/sleepedf_confusion.png"))
display(Markdown(open("results/sleepedf_results.md").read()))

## 8. Download the results
Commit `results/sleepedf_results.json`, `results/sleepedf_results.md`, `results/sleepedf_confusion.png`, `results/sleepedf_<model>_cv.json` and `results/latency_cpu_colab.json` to the repo, then paste the Markdown table into the README's Results section.

In [ ]:
!zip -q -r /content/neuraledge_sleepedf_results.zip results artifacts/sleepedf_{MODEL}.onnx artifacts/sleepedf_{MODEL}_fp32.pt
from google.colab import files
files.download("/content/neuraledge_sleepedf_results.zip")